In [1]:
!pip install pandas matplotlib numpy

In [25]:
import json
import pandas as pd

# ============================================================
# CONFIGURATION
# ============================================================

file_path = 'script_job_15bde4241f7c72f5e9c454b6efc1298a_0.json'

# ============================================================
# 1. LOAD JSON DATA
# ============================================================

try:
    with open(file_path, 'r') as f:
        data = json.load(f)
    print(f"Loaded {len(data)} time windows.")
except FileNotFoundError:
    print("ERROR: JSON file not found.")
    data = []

if not data:
    print("No data available.")
    exit()

# ============================================================
# 2. FLATTEN JSON STRUCTURE
# ============================================================

rows = []
for window in data:
    window_id = window.get('window_id')

    for pool in window.get('top_congested_pools', []):
        row = pool.copy()
        row['window_id'] = int(window_id) if window_id else 0
        rows.append(row)

if not rows:
    print("No pools found in JSON.")
    exit()

df = pd.DataFrame(rows)

# ============================================================
# 3. TYPE CONVERSION
# ============================================================

numeric_cols = [
    'total_tx_in_window',
    'total_failed_in_window',
    'slots_active_count',
    'p95_tx_per_slot',
    'p99_tx_per_slot',
    'p99_9_tx_per_slot'
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

# Shorten pair names
df['pair_label'] = df['token_mints'].apply(
    lambda x: x[:4] + '..' + x[-4:] if isinstance(x, str) else 'N/A'
)

# ============================================================
# 4. AGGREGATE BY DEX x PAIR x WINDOW
# ============================================================
# For each (DEX, PAIR, WINDOW) combination, aggregate all pools

window_agg = df.groupby(['window_id', 'dex_name', 'token_mints', 'pair_label']).agg({
    'total_tx_in_window': 'sum',           # Total TX across all pools
    'total_failed_in_window': 'sum',       # Total failed TX
    'pool_address': 'nunique',             # Number of pools (shards)
    # For percentiles: take MAX (worst single pool peak)
    # NOT sum - you cannot add percentiles mathematically!
    'p95_tx_per_slot': 'max',              # Worst P95 among all pools
    'p99_tx_per_slot': 'max',              # Worst P99 among all pools
    'p99_9_tx_per_slot': 'max'             # Worst P99.9 among all pools
}).reset_index()

# Calculate average TX per slot across ALL 2000 slots in window
window_agg['avg_tx_per_slot'] = window_agg['total_tx_in_window'] / 2000

# Calculate failure rate
window_agg['failure_rate_pct'] = (
    window_agg['total_failed_in_window'] /
    window_agg['total_tx_in_window'].replace(0, 1) * 100
)

# ============================================================
# 5. FIND PEAK WINDOW FOR EACH DEX x PAIR
# ============================================================

peak_indices = window_agg.groupby(['dex_name', 'token_mints'])['total_tx_in_window'].idxmax()
peaks_df = window_agg.loc[peak_indices].sort_values('total_tx_in_window', ascending=False)

# ============================================================
# 6. DEX SUMMARY
# ============================================================

dex_summary = peaks_df.groupby('dex_name').agg({
    'avg_tx_per_slot': 'mean',             # Average demand per slot
    'p99_tx_per_slot': 'mean',             # Average of worst P99 peaks
    'total_tx_in_window': 'mean',
    'failure_rate_pct': 'mean',
    'pool_address': 'mean',
    'token_mints': 'count'
}).reset_index()

dex_summary.columns = [
    'DEX',
    'Avg TX per Slot (all 2000)',
    'Avg P99 Peak (worst pool)',
    'Avg Total TX in Window',
    'Avg Failure Rate (%)',
    'Avg Number of Shards',
    'Number of Pairs'
]

dex_summary = dex_summary.sort_values('Avg TX per Slot (all 2000)', ascending=False)

# ============================================================
# 7. DETAILED TOP PAIRS TABLE
# ============================================================

top_pairs = peaks_df.head(20)[[
    'dex_name', 'pair_label', 'window_id', 'pool_address',
    'total_tx_in_window',
    'avg_tx_per_slot', 'failure_rate_pct',
    'p95_tx_per_slot', 'p99_tx_per_slot', 'p99_9_tx_per_slot'
]].copy()

top_pairs.columns = [
    'DEX', 'Pair', 'Window', 'Shards',
    'Total TX',
    'Avg TX/Slot', 'Fail %',
    'P95 Peak', 'P99 Peak', 'P99.9 Peak'
]

# ============================================================
# 8. REPORTS
# ============================================================

print("\n" + "=" * 120)
print("DEX SUMMARY: Average Metrics at Peak Load")
print("=" * 120)
print("\nMetrics explanation:")
print("  - Avg TX per Slot: Total TX across all pools / 2000 slots")
print("  - P99 Peak: Highest P99 value observed among pools (worst single pool)")
print("  - Failure Rate: % of transactions that failed")
print("  - Shards: Number of separate pools handling the same pair\n")

pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print(dex_summary.to_string(index=False))

print("\n" + "=" * 120)
print("TOP 20 PAIRS: Detailed Metrics at Peak Window")
print("=" * 120)
print("\nInterpretation:")
print("  - 'Avg TX/Slot' shows average demand spread across 2000 slots")
print("  - 'P99 Peak' shows the worst congestion spike in a single pool")
print("  - If Shards > 1: traffic was distributed; Single Vault would see Total TX\n")

print(top_pairs.to_string(index=False))

# ============================================================
# 9. SAVE RESULTS
# ============================================================

dex_summary.to_csv('dex_summary_2000slots.csv', index=False)
top_pairs.to_csv('top_pairs_2000slots.csv', index=False)

print("\n\nResults saved to:")
print("  - dex_summary_2000slots.csv")
print("  - top_pairs_2000slots.csv")

Loaded 407 time windows.

DEX SUMMARY: Average Metrics at Peak Load

Metrics explanation:
  - Avg TX per Slot: Total TX across all pools / 2000 slots
  - P99 Peak: Highest P99 value observed among pools (worst single pool)
  - Failure Rate: % of transactions that failed
  - Shards: Number of separate pools handling the same pair

            DEX  Avg TX per Slot (all 2000)  Avg P99 Peak (worst pool)  Avg Total TX in Window  Avg Failure Rate (%)  Avg Number of Shards  Number of Pairs
   Meteora DLMM                        6.21                      49.83                12421.12                 22.41                  1.53              329
   Raydium CPMM                        3.98                      38.42                 7952.92                 64.29                  1.38               64
 Orca Whirlpool                        3.96                      39.12                 7921.75                 32.20                  1.22              119
Meteora Dynamic                        2.71 